# Étape 1 - Chargement et nettoyage des données

Dans cette étape on récupère les données brutes des restaurants et on les transforme
en un jeu propre, prêt pour la modélisation.

## Ce que fait le pré-traitement

Il se déroule en 4 briques, déjà codées dans `foodcast/domain/` :

| Fonction | Rôle |
|---|---|
| `extract` | lit les fichiers CSV hebdomadaires d'un restaurant sur un intervalle de semaines |
| `clean` | nettoie : noms de colonnes en minuscules, `order_date` en datetime, calcule `cash_in` (montant par commande), supprime les colonnes inutiles, trie par date |
| `merge` | fusionne restaurant_1 + restaurant_2 en un seul dataframe (la chaîne) |
| `resample` | ré-échantillonne à la maille **1 heure** (somme du `cash_in` par heure) |

Ces 4 briques sont enchaînées par une fonction maître unique : **`etl`**.

## 1. Importer les librairies

In [7]:
import sys
sys.path.append('..')
import yaml
import logging
import logging.config
import numpy as np
import pandas as pd
pd.set_option('display.min_rows', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 500)
pd.set_option('max_colwidth', 400)

from foodcast.domain.transform import etl
from foodcast.domain.feature_engineering import features_offline, features_online
from foodcast.domain.forecast import span_future, cross_validate, plotly_predictions
from foodcast.domain.multi_model import MultiModel
from sklearn.ensemble import RandomForestRegressor
import foodcast.settings as settings
import plotly.graph_objects as go

with open(settings.LOGGING_CONFIGURATION_FILE, 'r') as f:
    logging.config.dictConfig(yaml.safe_load(f.read()))

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Regarder le code de `etl`

`??` affiche la signature, la docstring **et** le code source complet.

In [8]:
etl??

Signature: etl(data_dir: str, start_week: int, end_week: int) -> pandas.DataFrame
Source:   
@log_return_shape
def etl(data_dir: str, start_week: int, end_week: int) -> pd.DataFrame:
    """
    Load a cleaned temporal slice of data.

    Parameters
    ----------
    data_dir : str
        Data directory path.
    start_week : int
        First week number (included).
    end_week : int
        Last week number (included).

    Returns
    -------
    pd.DataFrame
        Cleaned data slice between start_week and end_week.
    """
    df1 = extract(data_dir, start_week, end_week, 'restaurant_1')
    df2 = extract(data_dir, start_week, end_week, 'restaurant_2')
    df1 = clean(df1)
    df2 = clean(df2)
    df = merge(df1, df2)
    df = resample(df)
    return df
File:      ~/Documents/ml_data_base/MlOps_1/foodcast/domain/transform.py
Type:      function

## 3. Extraire un jeu de données pré-traité (semaines 197 à 200)

Signature : `etl(data_dir, start_week, end_week)`.

- `data_dir` : le dossier des données, disponible dans `settings.DATA_DIR`
- `start_week` / `end_week` : numéros de semaine, **inclus** tous les deux

In [15]:
df = etl(settings.DATA_DIR, 197, 200)
df.head(20)

2026-09-09 15:04:30 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/infrastructure/extract.py - INFO - extract: shape = (2158, 6)
2026-09-09 15:04:30 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/infrastructure/extract.py - INFO - extract: shape = (3247, 6)
2026-09-09 15:04:30 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - clean: shape = (380, 3)
2026-09-09 15:04:30 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - clean: shape = (565, 3)
2026-09-09 15:04:30 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - INFO - merge: shape = (945, 2)
2026-09-09 15:04:30 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/decorators.py - INFO - resample: shape = (659, 2)
2026-09-09 15:04:30 - /Users/moussa/Documents/ml_data_base/MlOps_1/forecasting/../foodcast/domain/transform.py - I

,order_date,cash_in
0,2018-10-08 09:00:00,89.55
1,2018-10-08 10:00:00,0.00
2,2018-10-08 11:00:00,0.00
3,2018-10-08 12:00:00,0.00
4,2018-10-08 13:00:00,0.00
5,2018-10-08 14:00:00,0.00
6,2018-10-08 15:00:00,0.00
7,2018-10-08 16:00:00,52.55
8,2018-10-08 17:00:00,158.15
9,2018-10-08 18:00:00,391.45


Le dataframe obtenu a une ligne par heure, avec deux colonnes :

- `order_date` : l'horodatage (début de l'heure)
- `cash_in` : le chiffre d'affaires encaissé pendant cette heure (les deux restaurants réunis)

In [10]:
df.shape

(659, 2)

In [14]:
df.describe()

,order_date,cash_in
count,659,659.000000
mean,2018-10-22 02:00:00,48.434522
min,2018-10-08 09:00:00,0.000000
25%,2018-10-15 05:30:00,0.000000
50%,2018-10-22 02:00:00,0.000000
75%,2018-10-28 22:30:00,29.725000
max,2018-11-04 19:00:00,974.600000
std,NaN,123.104489


## 4. Tracer le chiffre d'affaires en fonction du temps

On utilise `plotly` : un objet `go.Figure()` auquel on ajoute une trace `go.Scatter`
(x = les dates, y = le chiffre d'affaires).

In [13]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=df['order_date'], y=df['cash_in'], name='cash-in')
)
fig.update_layout(
    title='Cash-in',
    xaxis_title='date',
    yaxis_title='dollars',
    font=dict(family='Computer Modern', size=18, color='#7f7f7f'),
)
fig.show()

**Ce qu'on observe :** une forte saisonnalité *journalière* (pics le soir, creux la nuit)
et *hebdomadaire* (week-ends plus chargés). C'est exactement ce que le feature engineering
de l'étape suivante va encoder.

➡️ Étape suivante : `02_feature_engineering_offline.ipynb`